In [49]:
import os
import re
import pandas as pd
from pymongo import MongoClient
from urllib.parse import quote_plus
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Load Environment & Connect to MongoDB

In [50]:
# Load environment variables
load_dotenv()

# Secure credentials
username = quote_plus(os.getenv("MONGO_ATLAS_USERNAME"))
password = quote_plus(os.getenv("MONGO_ATLAS_PASSWORD"))
cluster = os.getenv("MONGO_ATLAS_CLUSTER")
app_name = os.getenv("MONGO_ATLAS_APP_NAME", "app")

db_name = os.getenv("MONGO_DB_ATLAS")
collection_name = os.getenv("MONGO_MODIFIED_COLLECTION_ATLAS")

# MongoDB connection string
mongo_uri = (
    f"mongodb+srv://{username}:{password}@{cluster}/"
    f"{db_name}?retryWrites=true&w=majority&appName={app_name}"
)

# Connect to database
client = MongoClient(mongo_uri)
db = client[db_name]
collection = db[collection_name]

print(f"Connected to collection: {collection_name}")

Connected to collection: virus_total_modelado


# Feature Engineering Helpers

In [51]:
def extract_main_trid_type(trid_string):
    if not isinstance(trid_string, str) or not trid_string:
        return "Unknown"

    first_line = trid_string.split("\n")[0]
    return re.sub(r"\(.*\)", "", first_line).strip()

# Feature Extraction

In [52]:
# Antivirus labels to predict
target_antivirus = [
    "Sophos", "ESET-NOD32", "Avira",
    "Qihoo-360", "Cynet", "CAT-QuickHeal", "Alibaba"
]

dataset = []

cursor = collection.find({})

for doc in cursor:

    row = {
        "id": doc.get("_id"),
        "vhash": doc.get("vhash"),
        "main_type": extract_main_trid_type(doc.get("trid"))
    }

    # --- File extension features ---
    extensions = doc.get("extensions", {})
    row.update({
        "ext_dex": extensions.get("dex", 0),
        "ext_xml": extensions.get("xml", 0),
        "ext_js": extensions.get("js", 0),
        "ext_png": extensions.get("png", 0),
    })

    # --- Permission features (binary encoding) ---
    permissions = doc.get("androguard", {}).get("Permissions", {})
    row.update({
        "perm_boot": int("android.permission.RECEIVE_BOOT_COMPLETED" in permissions),
        "perm_phone": int("android.permission.READ_PHONE_STATE" in permissions),
        "perm_internet": int("android.permission.INTERNET" in permissions),
        "perm_storage": int("android.permission.WRITE_EXTERNAL_STORAGE" in permissions),
    })

    # --- Tags ---
    tags = doc.get("tags", [])
    row["is_apk"] = int("apk" in tags)

    # --- Target variables (multi-label classification) ---
    scans = doc.get("scans", {})
    for av in target_antivirus:
        row[f"target_{av}"] = int(scans.get(av, {}).get("detected", False))

    dataset.append(row)

# Convert to DataFrame
df = pd.DataFrame(dataset)

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (157, 19)


,id,vhash,main_type,ext_dex,ext_xml,ext_js,ext_png,perm_boot,perm_phone,perm_internet,perm_storage,is_apk,target_Sophos,target_ESET-NOD32,target_Avira,target_Qihoo-360,target_Cynet,target_CAT-QuickHeal,target_Alibaba
0,4cdc32fca1ffd8a0570b85141ccbc0a8c5250052562e27...,cae9663cc552d87c5379e5692f09b5c3,Android Package,3,2,7,23,1,1,1,1,1,1,1,1,0,1,1,0
1,44443d7162bc918a52a80274fe77cfc709c189a5b654f5...,cae9663cc552d87c5379e5692f09b5c3,Android Package,3,2,7,23,1,1,1,1,1,1,1,1,0,1,1,0
2,291cc28724e71db197586474d2fee3d4c1e407ce9cc021...,cae9663cc552d87c5379e5692f09b5c3,Android Package,3,2,7,23,1,1,1,1,1,1,1,1,0,1,1,0
3,c69c9ada25b8e94660f35f9bea35dbb54ba1ed1cdc7c08...,50381c5ac8405cfed4352b1cd1ece5bb,Android Package,1,175,0,250,1,1,1,1,1,1,1,1,1,1,1,0
4,34e1c03fc6192d4b201b80279e7cc0af738802d5d30924...,c8b27252ff66362ebf38c6c0d5414201,Android Package,1,16,0,387,1,1,1,1,1,0,0,1,1,1,1,0


# Data Overview

In [53]:
target_cols = [c for c in df.columns if c.startswith("target_")]

print("Positive detections per antivirus:")
print(df[target_cols].sum())

Positive detections per antivirus:
target_Sophos            93
target_ESET-NOD32       123
target_Avira            142
target_Qihoo-360         82
target_Cynet            140
target_CAT-QuickHeal    139
target_Alibaba           35
dtype: int64


# Train / Test Split

In [54]:
# Features (X)
feature_cols = [
    col for col in df.columns
    if col not in target_cols and col not in ["id", "vhash", "main_type"]
]

X = df[feature_cols]
y = df[target_cols]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (125, 9)
Test set: (32, 9)


# Model Training

In [55]:
# Base model
base_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Multi-label wrapper
model = MultiOutputClassifier(base_model)

# Train model
model.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


# Evaluation

In [56]:
# Predictions
y_pred = model.predict(X_test)

print("=== Classification Report per Antivirus ===")

for i, col in enumerate(target_cols):
    print(f"\n{col}")
    print(classification_report(y_test.iloc[:, i], y_pred[:, i]))

=== Classification Report per Antivirus ===

target_Sophos
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        13
           1       1.00      1.00      1.00        19

    accuracy                           1.00        32
   macro avg       1.00      1.00      1.00        32
weighted avg       1.00      1.00      1.00        32


target_ESET-NOD32
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         6
           1       1.00      1.00      1.00        26

    accuracy                           1.00        32
   macro avg       1.00      1.00      1.00        32
weighted avg       1.00      1.00      1.00        32


target_Avira
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         4
           1       1.00      1.00      1.00        28

    accuracy                           1.00        32
   macro avg       1.00      1.00   

# Global Accuracy

In [57]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Global subset accuracy: {accuracy:.2%}")

Global subset accuracy: 90.62%
